# Preparing Medical Fine-Tuning Data

In [33]:
from datasets import load_dataset
import pandas as pd
import json

In [2]:
## Download Dataset
ds = load_dataset("MegaScience/MegaScience")
ds.set_format("pandas")

README.md: 0.00B [00:00, ?B/s]

C:\Users\Colby\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Colby\.cache\huggingface\hub\datasets--MegaScience--MegaScience. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to re

train-00000-of-00001.parquet:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1253230 [00:00<?, ? examples/s]

In [ ]:
## Get 'train' split as pandas DataFrame
df = ds['train'].to_pandas()
df.shape

In [ ]:
## Subset to where 'subject' is 'medicine'
med_df = df[df['subject'] == 'medicine']
med_df.shape

(81638, 5)

In [32]:
med_df.head()

,question,answer,subject,reference_answer,source
0,In a patient diagnosed with a testicular germ ...,Elevated HCG levels without elevated AFP are s...,medicine,Suggestive of a seminoma.,textbook_reasoning
12,"What are the key epidemiologic factors (age, g...",The key epidemiologic factors that may suggest...,medicine,The key epidemiologic factors that may suggest...,textbook_reasoning
16,Why might additional knowledge about when duri...,The timing of haemorrhage during urination can...,medicine,Observing when haemorrhage occurs during urina...,textbook_reasoning
29,What is the treatment for dental disease in fe...,The treatment for dental disease in ferrets in...,medicine,"The treatment includes dental examination, cle...",textbook_reasoning
37,What imaging modalities are recommended for ev...,The following imaging modalities are recommend...,medicine,"MRI brain/orbits, Ocular ultrasound (US), CT c...",textbook_reasoning


In [36]:
## Split into training and validation sets
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(med_df, test_size=0.2, random_state=1337)

In [34]:
## Define JSONL formatting function
def format_jsonl(row):
    jsonl_row = {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant for answering medical questions."},
            {"role": "user", "content": row['question']},
            {"role": "assistant", "content": row['answer']}
        ]}

    return jsonl_row

In [37]:
## Loop through DataFrame and apply formatting function
train_jsonl = train_df.apply(format_jsonl, axis=1).tolist()
val_jsonl = val_df.apply(format_jsonl, axis=1).tolist()

train_jsonl[:2]
val_jsonl[:2]

[{'messages': [{'role': 'system',
    'content': 'You are a helpful assistant for answering medical questions.'},
   {'role': 'user',
    'content': 'When identifying biochemical isolates of *Shigella*, what precaution should be taken to avoid false-positive serological confirmation?'},
   {'role': 'assistant',
    'content': 'Care must be taken to perform biochemical identification before serological typing because *Shigella* isolates can cross-react with *E. coli* in serological tests. Some *E. coli* strains (e.g., anaerogenic, lactose-negative, or delayed lactose-fermenting strains) may resemble *Shigella* biochemically and serologically, leading to false-positive results if serotyping is done without prior biochemical confirmation.  \n\n$\\boxed{\\text{Biochemical identification should precede serological typing to avoid cross-reactions with } E. coli.}$'}]},
 {'messages': [{'role': 'system',
    'content': 'You are a helpful assistant for answering medical questions.'},
   {'role'

In [38]:
## Save JSONL data to files
with open('medical_training_set.jsonl', 'w') as f:
    for item in train_jsonl:
        f.write(json.dumps(item) + '\n')

with open('medical_validation_set.jsonl', 'w') as f:
    for item in val_jsonl:
        f.write(json.dumps(item) + '\n')

## Generate Mini Datasets for UI-based Fine-Tuning

In [39]:
## Loop through DataFrame and apply formatting function
train_mini_jsonl = train_df[:1000].apply(format_jsonl, axis=1).tolist()
val_mini_jsonl = val_df[:1000].apply(format_jsonl, axis=1).tolist()

train_mini_jsonl[:2]
val_mini_jsonl[:2]

[{'messages': [{'role': 'system',
    'content': 'You are a helpful assistant for answering medical questions.'},
   {'role': 'user',
    'content': 'When identifying biochemical isolates of *Shigella*, what precaution should be taken to avoid false-positive serological confirmation?'},
   {'role': 'assistant',
    'content': 'Care must be taken to perform biochemical identification before serological typing because *Shigella* isolates can cross-react with *E. coli* in serological tests. Some *E. coli* strains (e.g., anaerogenic, lactose-negative, or delayed lactose-fermenting strains) may resemble *Shigella* biochemically and serologically, leading to false-positive results if serotyping is done without prior biochemical confirmation.  \n\n$\\boxed{\\text{Biochemical identification should precede serological typing to avoid cross-reactions with } E. coli.}$'}]},
 {'messages': [{'role': 'system',
    'content': 'You are a helpful assistant for answering medical questions.'},
   {'role'

In [40]:
## Save JSONL data to files
with open('medical_training_set_mini.jsonl', 'w') as f:
    for item in train_mini_jsonl:
        f.write(json.dumps(item) + '\n')

with open('medical_validation_set_mini.jsonl', 'w') as f:
    for item in val_mini_jsonl:
        f.write(json.dumps(item) + '\n')